In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
!{sys.executable} -m pip install wordcloud
import os
import csv

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('punkt_tab')
from wordcloud import WordCloud



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Import Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.rename(columns={'Audio': 'song'}, inplace=True)
print("DataFrame 'df' loaded and 'Audio' column renamed to 'song'. First 5 rows:")
print(df.head())

DataFrame 'df' loaded and 'Audio' column renamed to 'song'. First 5 rows:
     ID  song                                           Response Duration  \
0  2001     1                   The sound of an elevator closing   500 ms   
1  2001     2           The sound of a door opened automatically   500 ms   
2  2001     5  The sound of a water faucet when it is turned ...   500 ms   
3  2001     6  The sound of small items like little beads bei...   500 ms   
4  2001     9  In a room where music is being produced or in ...  3000 ms   

  Timbre warble AttackTimes  Word_Count  \
0   sine   none        fast           6   
1   sine   none        slow           7   
2  noise   none        fast          13   
3  noise   none        slow          14   
4   sine   none        fast          15   

                                    Clean_Lemma_Text  
0                   the sound of an elevator closing  
1           the sound of a door opened automatically  
2  the sound of a water faucet when it 

## Clean Text Data

### remove capitalization and punctuation

In [ ]:
import sys
!{sys.executable} -m pip install pyspellchecker
print("pyspellchecker installed successfully.")

import re
import nltk

def clean_punc_case(text):
    text = str(text).lower()  # Convert to string and lowercase
    tokens = nltk.word_tokenize(text) # Tokenize
    # Remove non-alphabetic characters and join back
    cleaned_tokens = [word for word in tokens if word.isalpha()]
    return ' '.join(cleaned_tokens)

df['response_punc_case_cleaned'] = df['Response'].apply(clean_punc_case)
print("Created 'response_punc_case_cleaned' column. First 5 rows:")
print(df[['Response', 'response_punc_case_cleaned']].head())

from spellchecker import SpellChecker

spell = SpellChecker()

misspelled_words = set()

# Iterate through each entry in the cleaned text column
for text in df['response_punc_case_cleaned'].dropna():
    words = text.split()
    # Find words in the text that are not in the spell checker's vocabulary
    unknown_words = spell.unknown(words)
    misspelled_words.update(unknown_words)

print(f"Found {len(misspelled_words)} potential misspellings:")
print(sorted(list(misspelled_words)))
print("Note: This list contains words not found in the spell checker's dictionary. It may include proper nouns, domain-specific terms, or actual misspellings.")

pyspellchecker installed successfully.
Created 'response_punc_case_cleaned' column. First 5 rows:
                                            Response  \
0                   The sound of an elevator closing   
1           The sound of a door opened automatically   
2  The sound of a water faucet when it is turned ...   
3  The sound of small items like little beads bei...   
4  In a room where music is being produced or in ...   

                          response_punc_case_cleaned  
0                   the sound of an elevator closing  
1           the sound of a door opened automatically  
2  the sound of a water faucet when it is turned ...  
3  the sound of small items like little beads bei...  
4  in a room where music is being produced or in ...  
Found 132 potential misspellings:
['ailen', 'airpods', 'animalistic', 'app', 'applicance', 'apps', 'aux', 'bitklavier', 'cafe', 'chruch', 'cnn', 'comercial', 'dector', 'desmos', 'didn', 'dj', 'dont', 'doomscroll', 'echoey', 'edm', 'env

### correct misspelled words

In [ ]:
correction_map = {
    # clear typos
    'ailen': 'alien',
    'applicance': 'appliance',
    'chruch': 'church',
    'comercial': 'commercial',
    'dector': 'detector',
    'envountor': 'encounter',
    'excersize': 'exercise',
    'experiementing': 'experimenting',
    'flourescent': 'fluorescent',
    'furturistic': 'futuristic',
    'imaginging': 'imagining',
    'jsut': 'just',
    'ligth': 'light',
    'machineries': 'machinery',
    'mayve': 'maybe',
    'mircophone': 'microphone',
    'muisc': 'music',
    'notifcation': 'notification',
    'oscilating': 'oscillating',
    'perfomance': 'performance',
    'perfomers': 'performers',
    'qualityt': 'quality',
    'recieving': 'receiving',
    'resembalence': 'resemblance',
    'restraunt': 'restaurant',
    'reverbrated': 'reverberated',
    'semblence': 'semblance',
    'sniffiling': 'sniffling',
    'somehting': 'something',
    'someome': 'someone',
    'someting': 'something',
    'souns': 'sounds',
    'theramin': 'theremin',
    'tinitus': 'tinnitus',
    'vidogame': 'video game',
    'xylephone': 'xylophone',
    'handbell': 'hand bell',
    'handbells': 'hand bells',
    'windchime': 'wind chime',
    'germaphobe': 'germophobe',

    # contractions / informal fixes / abbreviations
    'didn': 'did not',
    'dont': 'do not',
    'im': 'i am',
    'isn': 'is not',
    'isnt': 'is not',
    'ive': 'i have',
    'thats': 'that is',
    'theres': 'there is',
    'playin': 'playing',
    'sth': 'something',
    'intro': 'introduction',

    # plural / spacing normalization
    'airpods': 'headphones',
    'apps': 'app',  # left unchanged (valid plural)
    'pdfs': 'pdfs',  # left unchanged
    'tvs': 'television',
    'xrays': 'x rays',
    'seatbelts': 'seat belts',
    'speedbumps': 'speed bumps',
    'soundcheck': 'sound check',
    'spacebar': 'space bar',
    'highschool': 'high school',
    'highspeed': 'high speed',
    'quadmates': 'roommates',
    'ufo': 'space ship',
    'teleports': 'teleport',
    'poofed': 'poof',
    'scooches': 'scooch',
    'scooching': 'scooch',

    # phonetic / near-miss
    'everytime': 'every time',
    'flatlining': 'flat-lining',
    'murry': 'murray',
    'satelite': 'satellite',

    # tech / product normalization
    'facetime': 'video call',
    'imessage': 'text message',
    'imovie': 'i movie',
    'flstudio': 'fl studio',
    'garageband': 'garage band',
    'macbook': 'laptop',
    'spotify': 'spotify',  # left unchanged
    'instagram': 'instagram',  # left unchanged
    'quizlet': 'quizlet',  # left unchanged
    'desmos': 'desmos',  # left unchanged
    'minecraft': 'minecraft',  # left unchanged
    'xbox': 'xbox',  # left unchanged
    'synth': 'synthesizer',
    'tech': 'technology',
    'ev': 'electric vehicle',
    'evs': 'electric vehicles',
    'sci': 'science fiction',

    # misc / sound-symbol normalization
    'echoey': 'echo',  # acceptable variant
    'tshhh': 'tsh',
    'ooo': 'oh',
}



def apply_corrections(text, correction_dict):
    words = text.split()
    corrected_words = [correction_dict.get(word, word) for word in words]
    return ' '.join(corrected_words)

df['response_spell_corrected'] = df['response_punc_case_cleaned'].apply(lambda x: apply_corrections(x, correction_map))
print("Created 'response_spell_corrected' column with sample corrections. First 5 rows:")
print(df[['response_punc_case_cleaned', 'response_spell_corrected']].head())

Created 'response_spell_corrected' column with sample corrections. First 5 rows:
                          response_punc_case_cleaned  \
0                   the sound of an elevator closing   
1           the sound of a door opened automatically   
2  the sound of a water faucet when it is turned ...   
3  the sound of small items like little beads bei...   
4  in a room where music is being produced or in ...   

                            response_spell_corrected  
0                   the sound of an elevator closing  
1           the sound of a door opened automatically  
2  the sound of a water faucet when it is turned ...  
3  the sound of small items like little beads bei...  
4  in a room where music is being produced or in ...  


### lemmatize text

In [ ]:
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet as wn
from nltk import word_tokenize, pos_tag
from collections import defaultdict

nltk.download('averaged_perceptron_tagger_eng', quiet=True) # Ensure resource is downloaded

# Re-define lemmatizer and tag_map if they are not globally accessible in the current execution context
# (They were defined in VwWhdCocEQ9m, assuming they persist or are re-defined as needed)
lemmatizer = nltk.stem.WordNetLemmatizer()
tag_map = defaultdict(lambda : wn.NOUN)
tag_map['J'] = wn.ADJ
tag_map['V'] = wn.VERB
tag_map['R'] = wn.ADV

def lemmatize_text(text):
    tokens = nltk.word_tokenize(str(text)) # Ensure text is string and tokenize
    culledwords = []
    for token, tag in pos_tag(tokens):
        # Apply specific tag adjustments as in VwWhdCocEQ9m
        if token.endswith('ing'):
            tag = 'VBG'
        if token == 'sings':
            tag = 'VBZ'
        # Lemmatize using the tag_map
        tlemma = lemmatizer.lemmatize(token, tag_map[tag[0]])
        culledwords.append(tlemma)
    return ' '.join(culledwords).strip().lower()

df['response_lemmatized'] = df['response_spell_corrected'].apply(lemmatize_text)

print("Created 'response_lemmatized' column. First 5 rows:")
print(df[['response_spell_corrected', 'response_lemmatized']].head())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Created 'response_lemmatized' column. First 5 rows:
                            response_spell_corrected  \
0                   the sound of an elevator closing   
1           the sound of a door opened automatically   
2  the sound of a water faucet when it is turned ...   
3  the sound of small items like little beads bei...   
4  in a room where music is being produced or in ...   

                                 response_lemmatized  
0                     the sound of an elevator close  
1             the sound of a door open automatically  
2  the sound of a water faucet when it be turn on...  
3  the sound of small item like little bead be po...  
4  in a room where music be be produce or in the ...  


In [ ]:
import nltk
from nltk.corpus import stopwords

nltk_stops = set(stopwords.words('english'))
custom_stopwords = {
    # Your custom list
    'music','context','remind','hear','reminds','remember','recall','think','memory','one','time','would','always','happen','associate','ago','specific','come','mind','since', 'play',
    'song','piece',"songs","album","imagine", "imagined", "sound","noise","sounds", "sounded", "story", "think", "picture", "reminded", "remind", "like", "remember", "something", "anything", "didn't", "could", "couldn't", "didnt", "someone", 'make'
}
# Lowercase all stopwords
all_stopwords = set(word.lower() for word in nltk_stops | custom_stopwords)

def remove_all_stopwords(text):
    words = text.split()
    # Using 'all_stopwords' which includes custom stopwords and NLTK stopwords
    filtered_words = [word for word in words if word not in all_stopwords]
    return ' '.join(filtered_words)

df['response_final_cleaned'] = df['response_lemmatized'].apply(remove_all_stopwords)

print("Created 'response_final_cleaned' column. First 5 rows:")
print(df[['response_lemmatized', 'response_final_cleaned']].head())

grouped_responses = df.groupby('song')['response_final_cleaned'].apply(lambda x: 'endofasubhere'.join(x.dropna())).reset_index()

print("Concatenated responses per song ('endofasubhere' separator). First 5 rows:")
print(grouped_responses.head())

df.rename(columns={'response_final_cleaned': 'cleaned_text'}, inplace=True)
print("Column 'response_final_cleaned' renamed to 'cleaned_text'. First 5 rows:")
print(df[['Response', 'cleaned_text']].head())

Created 'response_final_cleaned' column. First 5 rows:
                                 response_lemmatized  \
0                     the sound of an elevator close   
1             the sound of a door open automatically   
2  the sound of a water faucet when it be turn on...   
3  the sound of small item like little bead be po...   
4  in a room where music be be produce or in the ...   

                  response_final_cleaned  
0                         elevator close  
1                door open automatically  
2                      water faucet turn  
3  small item little bead pour container  
4      room produce introduction podcast  
Concatenated responses per song ('endofasubhere' separator). First 5 rows:
   song                             response_final_cleaned
0     1  elevator closeendofasubheregenuinely scenario ...
1     2  door open automaticallyendofasubheredifficulty...
2     5  water faucet turnendofasubheresweep sand round...
3     6  small item little bead pour co

### output processed/cleaned text dataframe for future use

In [ ]:
import os

# Construct the new filepath for the updated DataFrame
# Assuming DATA_PATH is '/content/drive/MyDrive/Single Sounds Study/Dataframes/Story_NLP_df.csv'
# We'll create a new file named Context_NLP_df_with_cleaned_text.csv in the same directory
output_dir = os.path.dirname(DATA_PATH)
output_filename = 'Context_NLP_df_with_cleaned_text_V2.csv'
output_filepath = os.path.join(output_dir, output_filename)

# Save the DataFrame with the new 'cleaned_text' column to CSV
df.to_csv(output_filepath, index=False)

print(f"DataFrame with 'cleaned_text' column saved to: {output_filepath}")

DataFrame with 'cleaned_text' column saved to: /content/drive/MyDrive/Single Sounds Study/Dataframes/Context_NLP_df_with_cleaned_text_V2.csv


## Create Wordclouds

In [ ]:
# --- WORDCLOUD SETTINGS ---
make_round = True
make_bw = True

wc_filepath = os.path.join(WORDCLOUD_DIR, WORDCLOUD_FILENAME)
max_tfidf_scaling = .45
max_subj_scaling = 35

# Create a circular mask if needed
x, y = np.ogrid[:300, :300]
mask = (x - 150) ** 2 + (y - 150) ** 2 > 130 ** 2
mask = 255 * mask.astype(int)

def my_tf_color_func(dictionary, freq_type):
    cmap = plt.colormaps.get_cmap('gist_rainbow')
    def inner(word, font_size, position, orientation, random_state=None, **kwargs):
        if freq_type == 'tfidf':
            scalefactor = 1 / max_tfidf_scaling
            rgb = cmap(dictionary[word] * scalefactor)
        elif freq_type == 'count':
            rgb = cmap(dictionary[word] / max_subj_scaling)
        return tuple(int(element * 255) for element in rgb)
    return inner

### according to stimulus

In [ ]:
# WORKING TFIDF PIPELINE
# --- GROUP BY AUDIO NUMBER ---
group_cols = ['song']  # Assuming 'song' is the audio identifier column

grouped = df.groupby('song')  # <- THIS WAS MISSING

# --- UPDATE WORDCLOUD FILENAME ---
WORDCLOUD_FILENAME = 'context_meam_wordclouds_by_audio_V2.pdf'

# --- WORDCLOUD SETTINGS --- SAME AS BEFORE
make_round = True
make_bw = True

wc_filepath = os.path.join(WORDCLOUD_DIR, WORDCLOUD_FILENAME)
max_tfidf_scaling = .45
max_subj_scaling = 35

# Create a circular mask if needed
x, y = np.ogrid[:300, :300]
mask = (x - 150) ** 2 + (y - 150) ** 2 > 130 ** 2
mask = 255 * mask.astype(int)

def my_tf_color_func(dictionary, freq_type):
    cmap = plt.colormaps.get_cmap('gist_rainbow')
    def inner(word, font_size, position, orientation, random_state=None, **kwargs):
        if freq_type == 'tfidf':
            scalefactor = 1 / max_tfidf_scaling
            rgb = cmap(dictionary[word] * scalefactor)
        elif freq_type == 'count':
            rgb = cmap(dictionary[word] / max_subj_scaling)
        return tuple(int(element * 255) for element in rgb)
    return inner

# --- MAIN WORDCLOUD PIPELINE ---
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize flag to check if any groups were processed
has_valid_groups = False

with PdfPages(wc_filepath) as pdf:
    for group_keys, group_df in grouped:
        # FIX 1: Handle single integer group keys
        if not isinstance(group_keys, tuple):
            group_keys = (group_keys,)  # Wrap single keys in a tuple

        # FIX 2: Clean group labels for file naming
        group_label = '_'.join(str(k).replace("/", "-") for k in group_keys)  # Handle special characters

        # Get the cleaned texts for the current group and ensure they are flattened into a single list of strings
        raw_group_texts = group_df['cleaned_text'].dropna().values.tolist()
        group_texts = []
        for item in raw_group_texts:
            if isinstance(item, list):
                group_texts.extend(item)
            else:
                group_texts.append(item)

        # Skip empty groups
        if not group_texts:
            print(f"Skipping empty group: {group_label}")
            continue

        has_valid_groups = True  # Mark that we have at least one valid group

        # --- SUBJECT COUNT DICTIONARY ---
        tokenized_bysub = [nltk.word_tokenize(text) for text in group_texts]
        all_words = [word for tokens in tokenized_bysub for word in tokens]
        subj_count_dict = {word: sum(word in tokens for tokens in tokenized_bysub) for word in set(all_words)}
        subj_count_df = pd.DataFrame.from_dict(subj_count_dict, orient='index', columns=['count'])
        subj_count_df = subj_count_df.sort_values(by='count', ascending=False).iloc[:30]
        subj_count_df.to_csv(os.path.join(WORDCLOUD_DIR, f'highest_subjcount_words_{group_label}.csv'), encoding='utf-8')

        # --- TFIDF DICTIONARY ---
        tfidf_vectorizer = TfidfVectorizer(
            analyzer='word',
            stop_words=list(all_stopwords),
            sublinear_tf=False,
            min_df=1,
            max_df=1 if len(group_texts) == 1 else 0.8
        )
        tfidf_matrix = tfidf_vectorizer.fit_transform(group_texts)
        tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
        tfidf_dict = dict(zip(tfidf_vectorizer.get_feature_names_out(), tfidf_scores))
        # Filter top 30 and those used by at least 2 subjects
        tfidf_dict = {k: v for k, v in sorted(tfidf_dict.items(), key=lambda item: item[1], reverse=True) if subj_count_dict.get(k, 0) >= 2}
        tfidf_dict = dict(list(tfidf_dict.items())[:30])
        pd.DataFrame.from_dict(tfidf_dict, orient='index', columns=['tfidf']).to_csv(
            os.path.join(WORDCLOUD_DIR, f'highest_tfidf_words_{group_label}.csv'),
            encoding='utf-8'
        )

        # --- PLOTTING WORDCLOUDS ---
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))

        # TFIDF WordCloud
        wc_args = dict(
            width=1200, height=600, random_state=22, background_color="white",
            stopwords=all_stopwords, collocations=False, relative_scaling=1, max_words=30,
            color_func=my_tf_color_func(tfidf_dict, 'tfidf')
        )
        if make_round:
            wc_args['mask'] = mask
        cloud_tfidf = WordCloud(**wc_args).generate_from_frequencies(tfidf_dict)
        ax[0].set_title(f"{group_label} tfidf")
        ax[0].axis("off")
        if make_bw:
            ax[0].imshow(cloud_tfidf.recolor(color_func=lambda *args, **kwargs: "black", random_state=3), interpolation="bilinear")
        else:
            im0 = ax[0].imshow(cloud_tfidf)
            divider0 = make_axes_locatable(ax[0])
            cax0 = divider0.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(im0, cax0)

        # Subject Count WordCloud
        wc_args['color_func'] = my_tf_color_func(subj_count_dict, 'count')
        cloud_subj = WordCloud(**wc_args).generate_from_frequencies(subj_count_dict)
        ax[1].set_title(f"{group_label} subjcount")
        ax[1].axis("off")
        if make_bw:
            ax[1].imshow(cloud_subj.recolor(color_func=lambda *args, **kwargs: "black", random_state=3), interpolation="bilinear")
        else:
            im1 = ax[1].imshow(cloud_subj)
            divider1 = make_axes_locatable(ax[1])
            cax1 = divider1.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(im1, cax1)

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

      # FIX 3: Handle empty PDF case properly
        if not has_valid_groups:
          if os.path.exists(wc_filepath):
            os.remove(wc_filepath)
          print(f"No valid groups found in {group_cols} - deleted empty PDF")
        else:
          print(f"Successfully created {wc_filepath}")

Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_audio_V2.pdf


### according to audio features

In [ ]:
# --- GROUP BY AUDIO FEATURES ---
group_cols = ['Duration', 'warble', 'Timbre', 'AttackTimes']  # Modified grouping columns
grouped = df.groupby(group_cols)  # Group by multiple features

# --- UPDATE WORDCLOUD FILENAME ---
WORDCLOUD_FILENAME = 'context_meam_wordclouds_by_features_V2.pdf'  # New filename

# --- WORDCLOUD SETTINGS --- SAME AS BEFORE
make_round = True
make_bw = True

wc_filepath = os.path.join(WORDCLOUD_DIR, WORDCLOUD_FILENAME)
max_tfidf_scaling = .45
max_subj_scaling = 35

# Create a circular mask if needed
x, y = np.ogrid[:300, :300]
mask = (x - 150) ** 2 + (y - 150) ** 2 > 130 ** 2
mask = 255 * mask.astype(int)

def my_tf_color_func(dictionary, freq_type):
    cmap = plt.colormaps.get_cmap('gist_rainbow')
    def inner(word, font_size, position, orientation, random_state=None, **kwargs):
        if freq_type == 'tfidf':
            scalefactor = 1 / max_tfidf_scaling
            rgb = cmap(dictionary[word] * scalefactor)
        elif freq_type == 'count':
            rgb = cmap(dictionary[word] / max_subj_scaling)
        return tuple(int(element * 255) for element in rgb)
    return inner

# --- MAIN WORDCLOUD PIPELINE ---
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize flag to check if any groups were processed
has_valid_groups = False

with PdfPages(wc_filepath) as pdf:
    for group_keys, group_df in grouped:
        # FIX 1: Handle single integer group keys
        if not isinstance(group_keys, tuple):
            group_keys = (group_keys,)  # Wrap single keys in a tuple

        # FIX 2: Clean group labels for file naming
        group_label = '_'.join(str(k).replace("/", "-") for k in group_keys)  # Handle special characters
        # Get the cleaned texts for the current group and ensure they are flattened into a single list of strings
        raw_group_texts = group_df['cleaned_text'].dropna().values.tolist()
        group_texts = []
        for item in raw_group_texts:
            if isinstance(item, list):
                group_texts.extend(item)
            else:
                group_texts.append(item)

        # Skip empty groups
        if not group_texts:
            print(f"Skipping empty group: {group_label}")
            continue

        has_valid_groups = True  # Mark that we have at least one valid group

        # --- SUBJECT COUNT DICTIONARY ---
        tokenized_bysub = [nltk.word_tokenize(text) for text in group_texts]
        all_words = [word for tokens in tokenized_bysub for word in tokens]
        subj_count_dict = {word: sum(word in tokens for tokens in tokenized_bysub) for word in set(all_words)}
        subj_count_df = pd.DataFrame.from_dict(subj_count_dict, orient='index', columns=['count'])
        subj_count_df = subj_count_df.sort_values(by='count', ascending=False).iloc[:30]
        subj_count_df.to_csv(os.path.join(WORDCLOUD_DIR, f'highest_subjcount_words_{group_label}.csv'), encoding='utf-8')

        # --- TFIDF DICTIONARY ---
        tfidf_vectorizer = TfidfVectorizer(
            analyzer='word',
            stop_words=list(all_stopwords),
            sublinear_tf=False,
            min_df=1,
            max_df=1 if len(group_texts) == 1 else 0.8
        )
        tfidf_matrix = tfidf_vectorizer.fit_transform(group_texts)
        tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
        tfidf_dict = dict(zip(tfidf_vectorizer.get_feature_names_out(), tfidf_scores))
        # Filter top 30 and those used by at least 2 subjects
        tfidf_dict = {k: v for k, v in sorted(tfidf_dict.items(), key=lambda item: item[1], reverse=True) if subj_count_dict.get(k, 0) >= 2}
        tfidf_dict = dict(list(tfidf_dict.items())[:30])
        pd.DataFrame.from_dict(tfidf_dict, orient='index', columns=['tfidf']).to_csv(
            os.path.join(WORDCLOUD_DIR, f'highest_tfidf_words_{group_label}.csv'),
            encoding='utf-8'
        )

        # --- PLOTTING WORDCLOUDS ---
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))

        # TFIDF WordCloud
        wc_args = dict(
            width=1200, height=600, random_state=22, background_color="white",
            stopwords=all_stopwords, collocations=False, relative_scaling=1, max_words=30,
            color_func=my_tf_color_func(tfidf_dict, 'tfidf')
        )
        if make_round:
            wc_args['mask'] = mask
        cloud_tfidf = WordCloud(**wc_args).generate_from_frequencies(tfidf_dict)
        ax[0].set_title(f"{group_label} tfidf")
        ax[0].axis("off")
        if make_bw:
            ax[0].imshow(cloud_tfidf.recolor(color_func=lambda *args, **kwargs: "black", random_state=3), interpolation="bilinear")
        else:
            im0 = ax[0].imshow(cloud_tfidf)
            divider0 = make_axes_locatable(ax[0])
            cax0 = divider0.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(im0, cax0)

        # Subject Count WordCloud
        wc_args['color_func'] = my_tf_color_func(subj_count_dict, 'count')
        cloud_subj = WordCloud(**wc_args).generate_from_frequencies(subj_count_dict)
        ax[1].set_title(f"{group_label} subjcount")
        ax[1].axis("off")
        if make_bw:
            ax[1].imshow(cloud_subj.recolor(color_func=lambda *args, **kwargs: "black", random_state=3), interpolation="bilinear")
        else:
            im1 = ax[1].imshow(cloud_subj)
            divider1 = make_axes_locatable(ax[1])
            cax1 = divider1.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(im1, cax1)

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

      # FIX 3: Handle empty PDF case properly
        if not has_valid_groups:
          if os.path.exists(wc_filepath):
            os.remove(wc_filepath)
          print(f"No valid groups found in {group_cols} - deleted empty PDF")
        else:
          print(f"Successfully created {wc_filepath}")

Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
Successfully created /content/drive/MyDrive/Single Sounds Study/wordclouds/context_meam_wordclouds_by_features_V2.pdf
